# DistilBERT Fine-Tuning — Review Sentiment (Milestone 2, Improved Model)Fine-tunes `distilbert-base-uncased` on the 3-class sentiment target(Rating 1-2 negative, 3 neutral, 4-5 positive), to be compared against theTF-IDF + LogisticRegression baseline (validation macro-F1 **0.6245**).**Runtime: set Runtime -> Change runtime type -> T4 GPU before running.**On CPU this takes ~4.5 hours; on a T4, ~20 minutes.

## 1. Environment

In [ ]:
!nvidia-smiimport torchprint("CUDA available:", torch.cuda.is_available())assert torch.cuda.is_available(), "Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU"

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

## 2. Get the codeThe repo is public, so clone it directly. `src/` is imported as a package,so the repo root must be the working directory.

In [ ]:
import osREPO = "Ecommerce-Review-Analytics"if not os.path.exists(REPO):    !git clone https://github.com/Mashi98/{REPO}.gitos.chdir(f"/content/{REPO}")!git log --oneline -3

## 3. Get the data`data/processed/*.csv` is gitignored (the dataset is not redistributed), so thesplits have to be brought in separately. Two options — **A** is repeatable acrosssessions, **B** is quicker for a one-off.### Option A: Google DriveUpload `train.csv` / `val.csv` / `test.csv` to a Drive folder once, then point`DRIVE_SPLITS` at it.

In [ ]:
USE_DRIVE = TrueDRIVE_SPLITS = "/content/drive/MyDrive/capstone/processed"if USE_DRIVE:    from google.colab import drive    drive.mount("/content/drive")    !mkdir -p data/processed && cp {DRIVE_SPLITS}/*.csv data/processed/

### Option B: direct upload (run only if you skipped Option A)

In [ ]:
# from google.colab import files# import shutil, os# os.makedirs("data/processed", exist_ok=True)# for name, data in files.upload().items():#     shutil.move(name, f"data/processed/{name}")

In [ ]:
import pandas as pdfrom src.models.baseline import load_split, LABEL_COLtrain_df, val_df, test_df = load_split("train"), load_split("val"), load_split("test")print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")print(train_df[LABEL_COL].value_counts().to_dict())

## 4. Fine-tuneSame splits, same row set and same 3-class target as the baseline. The model isfed **raw** `Review Text` rather than the lemmatized column — DistilBERT's ownsubword tokenizer uses the negations and function words that `preprocess()`strips. Class imbalance is handled with inverse-frequency weighted loss, mirroringthe baseline's `class_weight="balanced"`, so the comparison isolates the model.The best checkpoint is selected on **validation macro-F1**, not loss or accuracy.

In [ ]:
import src.models.transformer as Ttrainer = T.train_transformer(    train_df,    val_df,    use_class_weights=True,    fp16=True,           # safe on a T4    epochs=3,    batch_size=16,    learning_rate=2e-5,)

## 5. Validation results vs. the baseline

In [ ]:
from src.evaluation.metrics import classification_metrics, per_class_report, confusion_matrix_dffrom src.models.transformer import SENTIMENT_LABELS, prepare_frame, ID2LABELval_true = prepare_frame(val_df)["label"].map(ID2LABEL).to_list()val_pred = T.predict_labels(trainer, val_df)summary = classification_metrics(val_true, val_pred, SENTIMENT_LABELS)print("DistilBERT validation:", {k: round(v, 4) for k, v in summary.items()})print("Baseline (LogReg + class weights): accuracy 0.7951 | macro_f1 0.6245 | weighted_f1 0.8083")print(f"macro-F1 delta vs baseline: {summary['macro_f1'] - 0.6245:+.4f}")print("Per-class:"); print(per_class_report(val_true, val_pred, SENTIMENT_LABELS).to_string())print("Confusion matrix:"); print(confusion_matrix_df(val_true, val_pred, SENTIMENT_LABELS).to_string())

## 6. Test set — run ONCE, only after the model is finalThe test split has been untouched through all model selection. Run this onlyfor the final baseline-vs-DistilBERT number that goes in the report.

In [ ]:
RUN_TEST_EVAL = False   # flip to True only for the final reported resultif RUN_TEST_EVAL:    test_true = prepare_frame(test_df)["label"].map(ID2LABEL).to_list()    test_pred = T.predict_labels(trainer, test_df)    print("DistilBERT TEST:", {k: round(v, 4) for k, v in classification_metrics(test_true, test_pred, SENTIMENT_LABELS).items()})    print("Per-class:"); print(per_class_report(test_true, test_pred, SENTIMENT_LABELS).to_string())    print("Confusion matrix:"); print(confusion_matrix_df(test_true, test_pred, SENTIMENT_LABELS).to_string())

## 7. Error analysisMisclassified validation reviews, for the report's error-analysis section.Neutral (3-star) is the baseline's weakest class at F1 0.41 — check whetherDistilBERT actually recovers it or just shifts the errors elsewhere.

In [ ]:
errors = val_df.reset_index(drop=True).assign(true=val_true, pred=val_pred)errors = errors[errors["true"] != errors["pred"]]print(f"{len(errors)} misclassified of {len(val_df)} ({len(errors)/len(val_df):.1%})")print("most common confusions:")print(errors.groupby(["true", "pred"]).size().sort_values(ascending=False).to_string())for (t, p), group in errors.groupby(["true", "pred"]):    print(f"--- true={t} predicted={p} ---")    for text in group["Review Text"].head(3):        print(" *", str(text)[:200])

## 8. Save the fine-tuned model back to Drive

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/capstone/distilbert-sentiment"trainer.save_model(SAVE_DIR)trainer.processing_class.save_pretrained(SAVE_DIR)print("saved to", SAVE_DIR)

## 9. Record for the reproducibility checklistCopy these into the Milestone 2 / Milestone 3 appendix.

In [ ]:
import transformers, torch, datetimeprint("date:              ", datetime.date.today())print("gpu:               ", torch.cuda.get_device_name(0))print("torch:             ", torch.__version__)print("transformers:      ", transformers.__version__)print("model:             ", T.MODEL_NAME)print("max_length:        ", T.MAX_LENGTH)print("learning_rate:     ", trainer.args.learning_rate)print("batch_size:        ", trainer.args.per_device_train_batch_size)print("epochs:            ", trainer.args.num_train_epochs)print("weight_decay:      ", trainer.args.weight_decay)print("warmup_steps:      ", trainer.args.warmup_steps)print("seed:              ", trainer.args.seed)print("class weighting:    inverse-frequency, normalized to mean 1")print("model selection:    best val macro_f1")